# Cosine LR Scheduler with Warmup

**难度:** Medium

实现带线性预热的余弦学习率调度。

该调度器在预热阶段线性增加学习率，之后按余弦曲线衰减，广泛用于 Transformer 训练。

**签名:** `cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0) -> float`

**参数:**
- `step` — 当前训练步数
- `total_steps` — 总步数
- `warmup_steps` — 预热步数
- `max_lr`, `min_lr` — 峰值和最小学习率

**返回:** 浮点数学习率

**约束:**
- 预热阶段：从 0 线性增加到 max_lr
- 衰减阶段：`min_lr + 0.5*(max_lr-min_lr)*(1+cos(pi*progress))`

**提示:**
1. step < warmup_steps → lr = max_lr * step / warmup_steps
2. step >= total_steps → lr = min_lr
3. progress = (step - warmup_steps) / (total_steps - warmup_steps)
   → min_lr + 0.5*(max_lr-min_lr)*(1 + cos(π·progress))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写 Cosine LR Schedule ----
# 面试要点：这个函数不涉及 PyTorch 高级 API，纯数学计算
# 核心是三段式：warmup → cosine decay → constant min_lr

import math

def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    # ---- 阶段 1: 线性预热 ----
    # 面试考点：为什么需要 warmup？
    # 答：训练初期参数随机，梯度方差大，大学习率容易震荡甚至发散
    # 预热让 optimizer 先"试探"，等梯度稳定后再用大学习率
    if step < warmup_steps:
        return max_lr * step / warmup_steps

    # ---- 阶段 2: 训练结束 ----
    if step >= total_steps:
        return min_lr

    # ---- 阶段 3: 余弦退火 ----
    # progress: 0 → 1，表示衰减阶段的进度
    progress = (step - warmup_steps) / (total_steps - warmup_steps)

    # 面试考点：为什么用余弦而不是线性？
    # 答：余弦衰减前期快后期慢，让模型在 loss landscape 的平坦区域多停留
    # (1 + cos(π * progress)) 从 2 单调递减到 0
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [ ]:
# Demo
lrs = [cosine_lr_schedule(i, 100, 10, 0.001) for i in range(101)]
print(f'Start: {lrs[0]:.6f}, Warmup end: {lrs[10]:.6f}, Mid: {lrs[55]:.6f}, End: {lrs[100]:.6f}')

In [ ]:
from torch_judge import check
check('cosine_lr')

## 📝 核心思路总结

1. **Warmup 防止初期不稳定**：训练初期梯度方差大，线性预热让学习率从 0 慢慢增大
2. **余弦衰减比线性更好**：开始衰减快、后期衰减慢，给模型更多时间在最优附近微调
3. **progress 从 0→1 映射到 cos 从 1→-1**：lr 从 max_lr 平滑降到 min_lr